# 1D CNN-A — Walk-Forward Inference

Loads the trained autoencoder, scaler, and K-Means model, then iterates bar-by-bar
through a date range of your choice.  For every new 64-bar window it:

1. **Encodes** — compresses the window to a 32-number latent vector
2. **Decodes** — reconstructs the window from the latent vector
3. **Scores** — measures reconstruction error (MSE); high = unusual pattern
4. **Labels** — assigns the nearest K-Means cluster

A live-updating chart lets you watch the anomaly score, pattern images, and
cluster assignment evolve as each new bar slides in.

> **Prerequisites:** run `1dcnn_train.ipynb` (saves `model.pt` and `scaler.pkl`),
> then `latent_cluster.ipynb` (saves `kmeans.pkl`).

## 1. Imports

In [ ]:
import os
import time
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from IPython.display import clear_output

from config import Config
from data  import load_bars, clean_data, add_features, drop_feature_nans, load_scaler
from model import load_model, load_kmeans

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

`INFER_START` and `INFER_END` set the date range you want to analyse.  The notebook
loads the **entire** CSV first so that EMA and rolling-mean indicators have enough
history to warm up — bars before `INFER_START` are used only for indicator
calculation, not for producing inference windows.

In [ ]:
cfg = Config()

# ── Override any base config values here ──────────────────────────────────────
# cfg.SYMBOL   = "TSLA"
# cfg.MAX_BARS = None   # load all bars

globals().update(vars(cfg))

# ── Inference-specific parameters ─────────────────────────────────────────────
INFER_START   = "2023-01-01"   # first date to produce windows for (must be in the CSV)
INFER_END     = "2023-06-30"   # last date to produce windows for
DISPLAY_EVERY = 20             # redraw the live chart every N windows (lower = slower but smoother)
STEP_DELAY    = 0.0            # seconds to pause after each redraw (0 = run as fast as possible)

print(f"Symbol : {SYMBOL}  |  Timeframe : {TIMEFRAME}")
print(f"Infer  : {INFER_START} → {INFER_END}")
print(f"Update every {DISPLAY_EVERY} windows, {STEP_DELAY}s delay")

## 3. Load Model, Scaler, and K-Means

All three were saved by earlier notebooks:

| File | Saved by | Used for |
|------|----------|---------|
| `model.pt` | `1dcnn_train.ipynb` | encode + decode each window |
| `scaler.pkl` | `1dcnn_train.ipynb` | normalise features the same way as training |
| `kmeans.pkl` | `latent_cluster.ipynb` | assign a cluster label to each latent vector |

If any file is missing, run the prerequisite notebook first.

In [ ]:
model  = load_model(DATA_DIR, SYMBOL, len(feature_cols), LATENT_DIM, DEVICE)
scaler = load_scaler(DATA_DIR, SYMBOL)
kmeans = load_kmeans(DATA_DIR, SYMBOL)

N_CLUSTERS = kmeans.n_clusters
print(f"Ready — {N_CLUSTERS} clusters, LATENT_DIM={LATENT_DIM}")

## 4. Load and Prepare Inference Data

### Why load the whole CSV?

Technical indicators need historical bars to warm up:
- EMA-50 needs at least 50 bars before its value is stable
- The 20-bar rolling mean for `volume_ratio` needs 20 bars
- Plus WINDOW_SIZE=64 bars to form the very first window

**Strategy:** load the entire CSV (or MAX_BARS), compute features on all of it,
apply the *training* scaler, then walk only the bars from `INFER_START` onward.

### Why use the saved scaler?

The autoencoder was trained on features normalised with a specific `RobustScaler`
fitted on the training data.  If we fit a fresh scaler on the inference data, the
feature values shift to a different numeric range — the model has never seen those
values and will produce nonsense output.  `scaler.transform()` applies the existing
fit; `scaler.fit_transform()` would create a brand-new fit (wrong here).

In [ ]:
# Load the full dataset — all bars are needed so indicator warm-up is correct.
df = load_bars(DATA_DIR, SYMBOL, TIMEFRAME, MAX_BARS)
df = clean_data(df)
df = add_features(df)
df = drop_feature_nans(df)

# CSV timestamps are stored as UTC-aware (datetime64[ns, UTC]).
# Strip the timezone so comparisons with plain date strings like "2023-01-01" work.
if df["timestamp"].dt.tz is not None:
    df["timestamp"] = df["timestamp"].dt.tz_convert(None)

# Apply the TRAINING scaler — same normalisation the model was trained on.
# .transform() applies the existing fit; do NOT call .fit_transform() here.
df[feature_cols] = scaler.transform(df[feature_cols])

# Find the row range that falls inside INFER_START..INFER_END.
infer_start_ts = pd.Timestamp(INFER_START)
infer_end_ts   = pd.Timestamp(INFER_END)
mask           = (df["timestamp"] >= infer_start_ts) & (df["timestamp"] <= infer_end_ts)
infer_rows     = df.index[mask]

if len(infer_rows) == 0:
    raise ValueError(
        f"No bars found between {INFER_START} and {INFER_END}.  "
        "Check that the dates exist in the CSV and that INFER_START > first bar + warm-up."
    )

# The first window that ends inside INFER_START needs WINDOW_SIZE prior bars.
# first_end_idx = the row index of the LAST bar in the first valid window.
first_end_idx = int(infer_rows[0]) + WINDOW_SIZE - 1
last_end_idx  = int(infer_rows[-1])

# Validate that we actually have enough history before the start date.
if first_end_idx >= len(df):
    raise ValueError(
        f"Not enough bars after {INFER_START} to form a full {WINDOW_SIZE}-bar window."
    )
if first_end_idx - WINDOW_SIZE + 1 < 0:
    raise ValueError(
        f"Not enough warm-up bars before {INFER_START}.  "
        f"Load more history or move INFER_START later."
    )

n_windows = last_end_idx - first_end_idx + 1
print(f"Total bars loaded : {len(df):,}")
print(f"Inference range   : {df['timestamp'].iloc[first_end_idx]}  →  {df['timestamp'].iloc[last_end_idx]}")
print(f"Windows to process: {n_windows:,}")

## 5. Live Display Helper

`_draw_live()` is called inside the loop every `DISPLAY_EVERY` steps to update
the chart in-place.  `clear_output(wait=True)` wipes the previous chart before
drawing the next one, creating a smooth animation effect.

The chart has five panels:

| Panel | What it shows |
|-------|--------------|
| **Top row (wide)** | Running MSE time series — the "anomaly score" over time |
| **Top right** | Current bar's raw OHLCV values + cluster label |
| **Bottom left** | Current 64×14 window as a greyscale image |
| **Bottom middle** | Latent vector (32 values) as a bar chart |
| **Bottom right** | Cluster history — colour strip of how cluster assignment changes |

In [ ]:
# Colour palette for clusters — up to 10 distinct colours.
_CLUSTER_COLOURS = plt.cm.tab10.colors

def _draw_live(
    ts,          # list of timestamps collected so far
    mse,         # list of MSE scores collected so far
    labels,      # list of cluster labels collected so far
    current_bar, # the current DataFrame row (pd.Series)
    window_np,   # current window array (WINDOW_SIZE, n_features) float32
    z,           # current latent vector (LATENT_DIM,) float32
    label,       # current cluster label (int)
    mse_now,     # current MSE value (float)
    feature_cols,
    n_clusters,
):
    """Render the 5-panel live inference dashboard."""
    fig = plt.figure(figsize=(20, 8))
    gs  = gridspec.GridSpec(2, 3, figure=fig, width_ratios=[2, 1, 1], hspace=0.4, wspace=0.35)

    # ── Top-left (wide): Running MSE time series ──────────────────────────────
    ax_mse = fig.add_subplot(gs[0, :2])
    ax_mse.plot(ts, mse, color="steelblue", linewidth=0.8, alpha=0.8)
    if len(mse) > 1:
        # Mark the 95th-percentile threshold so you can see what counts as unusual.
        p95 = float(np.percentile(mse, 95))
        ax_mse.axhline(p95, color="tomato", linestyle="--", linewidth=0.8, alpha=0.7,
                       label=f"p95 = {p95:.3f}")
        ax_mse.legend(fontsize=8, loc="upper left")
    # Highlight the current point.
    if ts:
        ax_mse.scatter([ts[-1]], [mse[-1]], color="tomato", s=30, zorder=5)
    ax_mse.set_title(
        f"Reconstruction Error (MSE)  —  current: {mse_now:.4f}  |  "
        f"Cluster {label}  |  window {len(ts):,} / {n_windows:,}",
        fontsize=10,
    )
    ax_mse.set_ylabel("MSE"); ax_mse.grid(alpha=0.2)
    ax_mse.tick_params(axis="x", labelsize=7, rotation=30)

    # ── Top-right: Current bar info ───────────────────────────────────────────
    ax_bar = fig.add_subplot(gs[0, 2])
    ax_bar.axis("off")
    ts_str = current_bar["timestamp"].strftime("%Y-%m-%d %H:%M") if hasattr(current_bar["timestamp"], "strftime") else str(current_bar["timestamp"])
    info_lines = [
        f"Timestamp : {ts_str}",
        f"Open      : {current_bar.get('open', float('nan')):.4f}",
        f"High      : {current_bar.get('high', float('nan')):.4f}",
        f"Low       : {current_bar.get('low', float('nan')):.4f}",
        f"Close     : {current_bar.get('close', float('nan')):.4f}",
        f"Volume    : {current_bar.get('volume', float('nan')):.0f}",
        "",
        f"MSE now   : {mse_now:.5f}",
        f"Cluster   : {label}",
    ]
    colour = _CLUSTER_COLOURS[label % len(_CLUSTER_COLOURS)]
    ax_bar.text(0.05, 0.95, "
".join(info_lines),
                transform=ax_bar.transAxes, fontsize=9, fontfamily="monospace",
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor=colour, alpha=0.25))
    ax_bar.set_title("Current Bar", fontsize=10)

    # ── Bottom-left: Current window as greyscale image ────────────────────────
    ax_win = fig.add_subplot(gs[1, 0])
    lo_w = window_np.min(); hi_w = window_np.max()
    rng  = hi_w - lo_w if hi_w > lo_w else 1.0
    win_u8 = ((window_np - lo_w) / rng * 255).astype(np.uint8)
    ax_win.imshow(win_u8, cmap="gray", aspect="auto", vmin=0, vmax=255,
                  interpolation="nearest")
    ax_win.set_xlabel("Feature channel", fontsize=8)
    ax_win.set_ylabel("Bar (newest at bottom)", fontsize=8)
    ax_win.set_xticks(range(len(feature_cols)))
    ax_win.set_xticklabels(feature_cols, rotation=90, fontsize=6)
    ax_win.set_title(f"Current 64-bar Window (greyscale)", fontsize=10)

    # ── Bottom-middle: Latent vector bar chart ────────────────────────────────
    ax_z = fig.add_subplot(gs[1, 1])
    colours_z = ["tomato" if v < 0 else "steelblue" for v in z]
    ax_z.bar(range(len(z)), z, color=colours_z, width=0.8)
    ax_z.axhline(0, color="black", linewidth=0.5)
    ax_z.set_xlabel("Latent dimension", fontsize=8)
    ax_z.set_ylabel("Activation", fontsize=8)
    ax_z.set_title(f"Latent Vector (dim={len(z)})", fontsize=10)
    ax_z.grid(alpha=0.2)

    # ── Bottom-right: Cluster history colour strip ────────────────────────────
    ax_cls = fig.add_subplot(gs[1, 2])
    if labels:
        # Draw each cluster label as a coloured bar — newest at the bottom.
        n_show = min(len(labels), 200)   # show the most recent 200 windows
        recent = labels[-n_show:]
        for j, lbl in enumerate(recent):
            c = _CLUSTER_COLOURS[lbl % len(_CLUSTER_COLOURS)]
            ax_cls.barh(j, 1, color=c, height=1.0, left=0)
        ax_cls.set_xlim(0, 1); ax_cls.set_yticks([])
        ax_cls.set_xlabel("Cluster", fontsize=8)
        ax_cls.set_title(f"Cluster History (last {n_show})", fontsize=10)
        # Add a mini legend for the cluster colours.
        from matplotlib.patches import Patch
        seen = sorted(set(recent))
        handles = [Patch(facecolor=_CLUSTER_COLOURS[c % len(_CLUSTER_COLOURS)],
                         label=f"C{c}") for c in seen]
        ax_cls.legend(handles=handles, fontsize=6, loc="upper right",
                      ncol=2, framealpha=0.7)

    plt.suptitle(
        f"Walk-Forward Inference — {SYMBOL}  {TIMEFRAME}",
        fontsize=12, fontweight="bold",
    )
    plt.show()

## 6. Walk-Forward Loop

Each iteration moves forward by **one bar**.  The 64-bar window slides:
- Bar that just dropped off: the oldest bar exits the left side of the window
- Bar that just arrived: the newest bar enters the right side of the window

The model sees only the current window — it has no memory of previous windows.
The "memory" is built by collecting `result_mse`, `result_labels`, etc. as we go.

In [ ]:
# Storage for results — one entry per window.
result_ts     = []   # timestamp of the final bar in each window
result_mse    = []   # reconstruction error (float)
result_labels = []   # K-Means cluster label (int)
result_z      = []   # latent vectors (np.ndarray, shape LATENT_DIM each)

model.eval()   # disable any training-mode behaviour (e.g. dropout)

for step, i in enumerate(range(first_end_idx, last_end_idx + 1)):

    # ── Extract the current window ────────────────────────────────────────────
    # The window is the WINDOW_SIZE bars that end at row i (inclusive).
    window_np = df[feature_cols].iloc[i - WINDOW_SIZE + 1 : i + 1].to_numpy(dtype=np.float32)
    # window_np shape: (WINDOW_SIZE, n_features)  e.g. (64, 14)

    # ── Convert to PyTorch tensor ─────────────────────────────────────────────
    # Conv1d expects channels-first: (batch, n_features, WINDOW_SIZE).
    # .T transposes (64, 14) → (14, 64), then .unsqueeze(0) adds the batch dim.
    window_t = torch.tensor(window_np).T.unsqueeze(0).to(DEVICE)
    # window_t shape: (1, 14, 64)

    # ── Run through the model ─────────────────────────────────────────────────
    with torch.no_grad():   # no_grad saves memory — we don't need gradients for inference
        z_t     = model.encoder(window_t)   # (1, LATENT_DIM)  — compressed representation
        recon_t = model.decoder(z_t)        # (1, n_features, WINDOW_SIZE) — reconstruction

    # ── Extract numpy arrays ──────────────────────────────────────────────────
    z     = z_t.cpu().numpy()[0]            # (LATENT_DIM,)
    recon = recon_t.cpu().numpy()[0].T      # (WINDOW_SIZE, n_features) — transpose back

    # ── Compute reconstruction error ──────────────────────────────────────────
    # Mean squared difference between original and reconstruction across all
    # time steps and feature channels.  Higher = the model found this window unusual.
    mse = float(((window_np - recon) ** 2).mean())

    # ── Assign cluster label ──────────────────────────────────────────────────
    # kmeans.predict() needs a 2D input: (1 sample, LATENT_DIM features).
    label = int(kmeans.predict(z.reshape(1, -1))[0])

    # ── Store results ─────────────────────────────────────────────────────────
    result_ts.append(df["timestamp"].iloc[i])
    result_mse.append(mse)
    result_labels.append(label)
    result_z.append(z)

    # ── Live display ──────────────────────────────────────────────────────────
    is_last = (step == n_windows - 1)
    if step % DISPLAY_EVERY == 0 or is_last:
        clear_output(wait=True)
        _draw_live(
            ts=result_ts,
            mse=result_mse,
            labels=result_labels,
            current_bar=df.iloc[i],
            window_np=window_np,
            z=z,
            label=label,
            mse_now=mse,
            feature_cols=feature_cols,
            n_clusters=N_CLUSTERS,
        )
        if STEP_DELAY > 0 and not is_last:
            time.sleep(STEP_DELAY)

print(f"Loop complete — processed {len(result_ts):,} windows.")

## 7. Summary

After the loop runs you have three arrays:
- `result_ts` — timestamp of each window's final bar
- `result_mse` — reconstruction error for each window
- `result_labels` — cluster label for each window

The cells below turn those into summary visualisations.

In [ ]:
# ── Convert to arrays for easy analysis ──────────────────────────────────────
ts_arr  = np.array(result_ts)
mse_arr = np.array(result_mse)
lbl_arr = np.array(result_labels)
Z_arr   = np.vstack(result_z)   # (n_windows, LATENT_DIM)

print(f"Windows processed : {len(mse_arr):,}")
print(f"MSE  — mean={mse_arr.mean():.4f}  std={mse_arr.std():.4f}  "
      f"min={mse_arr.min():.4f}  max={mse_arr.max():.4f}")
print(f"Clusters seen: {sorted(set(lbl_arr.tolist()))}")

In [ ]:
# ── Full MSE time series ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 4))
ax.plot(ts_arr, mse_arr, color="steelblue", linewidth=0.6, alpha=0.8)
p95 = float(np.percentile(mse_arr, 95))
ax.axhline(p95, color="tomato", linestyle="--", linewidth=1,
           label=f"95th percentile ({p95:.4f})")
ax.fill_between(ts_arr, mse_arr, p95,
                where=(mse_arr > p95), color="tomato", alpha=0.3,
                label="above p95 (unusual windows)")
ax.set_ylabel("Reconstruction MSE"); ax.grid(alpha=0.2)
ax.legend(fontsize=9)
ax.set_title(f"Full Reconstruction Error Timeline — {SYMBOL}  {INFER_START} to {INFER_END}")
plt.tight_layout(); plt.show()

In [ ]:
# ── Cluster distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart — fraction of time in each cluster
unique_labels, counts = np.unique(lbl_arr, return_counts=True)
colours = [_CLUSTER_COLOURS[c % len(_CLUSTER_COLOURS)] for c in unique_labels]
axes[0].pie(counts, labels=[f"Cluster {c}" for c in unique_labels],
            colors=colours, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Time Fraction per Cluster")

# Cluster label over time — colour scatter
for c in unique_labels:
    mask = lbl_arr == c
    axes[1].scatter(ts_arr[mask], lbl_arr[mask],
                    color=_CLUSTER_COLOURS[c % len(_CLUSTER_COLOURS)],
                    s=2, alpha=0.5, label=f"C{c}")
axes[1].set_yticks(unique_labels)
axes[1].set_ylabel("Cluster"); axes[1].grid(alpha=0.2)
axes[1].set_title("Cluster Assignment Over Time")
axes[1].tick_params(axis="x", labelsize=7, rotation=30)

plt.tight_layout(); plt.show()

In [ ]:
# ── Top 10 highest-anomaly windows ───────────────────────────────────────────
top_n = 10
top_idx = np.argsort(mse_arr)[-top_n:][::-1]   # indices of highest MSE

print(f"Top {top_n} most unusual windows:")
print(f"{'Rank':<5} {'Timestamp':<22} {'MSE':>8} {'Cluster':>8}")
print("-" * 47)
for rank, idx in enumerate(top_idx, 1):
    print(f"{rank:<5} {str(ts_arr[idx]):<22} {mse_arr[idx]:>8.5f} {lbl_arr[idx]:>8}")

In [ ]:
# ── Latent space heatmap (windows × latent dims) ─────────────────────────────
# Shows which latent dimensions are most active over the inference period.
# Each row is one window; each column is one latent dimension.
# Bright = high activation, dark = low activation.
fig, ax = plt.subplots(figsize=(20, 6))
img = ax.imshow(Z_arr.T, aspect="auto", cmap="RdBu_r",
                vmin=-3, vmax=3, interpolation="nearest")
plt.colorbar(img, ax=ax, label="Activation (clipped at ±3)")
ax.set_xlabel("Window index (time →)", fontsize=9)
ax.set_ylabel("Latent dimension", fontsize=9)
ax.set_title(
    f"Latent Space Heatmap — {Z_arr.shape[0]:,} windows × {Z_arr.shape[1]} dims  "
    f"({INFER_START} to {INFER_END})",
    fontsize=11,
)
plt.tight_layout(); plt.show()

## 8. What You're Seeing

### Reconstruction Error (MSE)
Every 64-bar window gets an MSE score — the average squared difference between the
original and the decoder's attempt to reconstruct it.

- **Low MSE** (near the mean): the model has seen many windows like this one — it's a
  familiar pattern.
- **High MSE** (above the 95th percentile): the model can't reconstruct this window
  well — it looks unusual compared to what it was trained on.  This could mean:
  - An earnings release, news event, or gap open
  - A very low-liquidity period (pre-market, extended hours)
  - Genuine price action the model hasn't been trained to recognise

### Latent Vector (32 bars)
Each bar shows how strongly one of the 32 "latent dimensions" fired for the current
window.  These dimensions have no explicit meaning — the model decided what each one
represents during training.  But patterns often emerge: some dimensions track
momentum, others track volatility, others fire only during specific session times.

### Cluster Label
The K-Means model grouped all training windows into N clusters by latent similarity.
Each cluster represents a type of market behaviour the model found.  The cluster
history strip shows how the dominant regime changes over time.

### Latent Space Heatmap (Section 7)
Reading across any row tells you the full latent "fingerprint" of that window.
Reading down any column tells you when that latent dimension was most active.
Columns that are mostly flat aren't doing much; columns with large variation are
the dimensions that matter most for distinguishing patterns.